# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
import pandas as pd
!git clone https://github.com/muhammadhuzaifanaeem/FlyRank-Internship.git


df = pd.read_csv('FlyRank-Internship/data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'avg_position', 'ctr',
                 'engagement_rate', 'scroll_rate', 'content_age_days',
                 'days_since_last_update', 'word_count']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_id']))
train_g, test_g = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42)
rf.fit(train_g[feature_cols].fillna(0), train_g['is_declining_label'])
test_g['model_score'] = rf.predict_proba(test_g[feature_cols].fillna(0))[:, 1]

def get_reason_codes(row):
    reasons = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append('stale_visible_page')
    if row['trend_direction'] == 'down' and row['impressions_90d'] >= 100:
        reasons.append('declining_with_demand')
    if pd.notna(row['word_count']) and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append('thin_visible_page')
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append('low_ctr_visible_page')
    if not reasons:
        reasons.append('general_refresh_review')
    return '|'.join(reasons)

test_g['reason_codes'] = test_g.apply(get_reason_codes, axis=1)
test_g['action_queue_rank'] = test_g['model_score'].rank(method='first', ascending=False).astype(int)

action_queue = test_g.sort_values('action_queue_rank')[
    ['content_id', 'client_id', 'action_queue_rank', 'model_score', 'reason_codes',
     'impressions_90d', 'avg_position', 'days_since_last_update']
]

print(action_queue.head(20))

fatal: destination path 'FlyRank-Internship' already exists and is not an empty directory.
                 content_id          client_id  action_queue_rank  \
11061  content_0b47dae0c7f9  client_8527a891e2                  1   
6957   content_a662ef2af9b4  client_8527a891e2                  2   
6228   content_e988c1699454  client_8527a891e2                  3   
28718  content_ef6e7d7cfe15  client_8527a891e2                  4   
20736  content_41baf0722ad9  client_8527a891e2                  5   
22524  content_846bb4dd8b44  client_8527a891e2                  6   
10080  content_35d63627bf3e  client_8527a891e2                  7   
25063  content_29884c0f9255  client_8527a891e2                  8   
5011   content_c148e44db30d  client_8527a891e2                  9   
12472  content_884c401ce126  client_8527a891e2                 10   
13561  content_685d3fea9361  client_8527a891e2                 11   
29629  content_643b51c0a848  client_8527a891e2                 12   
21926  conte

The action queue ranks pages by model score, from most to least urgent, with every page carrying a reason code explaining why it was flagged — so a reviewer never has to trust a black-box number.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

ntended use: This queue is meant for a content/SEO reviewer deciding which of many pages to check first when time is limited. It ranks pages by likelihood of needing review, based on patterns observed in this dataset.

Limits — where this stops being valid:

This is decision-support, not a directive. A high rank means "worth checking," never "definitely broken."
The model was trained and tested only on the clients in this dataset — for a brand-new client with very different content patterns, confidence should be lower.
This does not account for business context the data doesn't capture (e.g., a page intentionally kept static, seasonal content, legal/compliance pages).
This queue reflects a snapshot in time — it will get stale as real search performance shifts.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any flagged page, a human must check:

Is this page intentionally static (e.g. a legal page, a glossary, an evergreen reference)? If so, skip it regardless of rank.
Does the reason code actually match what a quick manual look shows? (e.g. does low_ctr_visible_page genuinely look like a title/snippet problem?)
Is there a business reason this page's metrics look this way that the data wouldn't capture (a recent campaign, a known outage, a redesign in progress)?

What should never be automated from this queue:

Automatically editing, publishing, or removing content based on rank alone.
Using this queue to make client-facing promises about ranking or traffic outcomes — this tool ranks review priority, not guaranteed results.
Treating a low rank as proof a page is fine — a low rank means "less urgent by this model," not "verified healthy."

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [3]:
# A monitoring signal: how much has the client mix or feature distribution shifted since training?
current_client_count = df['client_id'].nunique()
current_avg_staleness = df['days_since_last_update'].mean()

print(f"Baseline client count at training time: {train_g['client_id'].nunique()}")
print(f"Current client count: {current_client_count}")
print(f"Baseline avg days-since-update: {train_g['days_since_last_update'].mean():.1f}")
print(f"Current avg days-since-update: {current_avg_staleness:.1f}")

Baseline client count at training time: 24
Current client count: 32
Baseline avg days-since-update: 50.0
Current avg days-since-update: 46.1


Retrain/review triggers:

If the number of active clients grows or shrinks significantly from what the model was trained on, re-evaluate before trusting rankings for new clients.
If precision@50 on a fresh holdout drops meaningfully below the [your actual Week-6 number] baseline, retrain.
If the average "days since update" across all pages shifts significantly (a sign the underlying content operations changed), the freshness-based reason codes may need re-calibration.
Re-run the leakage audit (ML-09) any time new features are added to catch new leakage risks early.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
import os
os.makedirs('work/outputs', exist_ok=True)

# Save the action queue
action_queue.to_csv('work/outputs/action_queue.csv', index=False)

# Save a small metrics summary (safe to commit — no raw data)
import json
metrics_summary = {
    'total_pages_ranked': len(action_queue),
    'unique_clients_in_queue': int(test_g['client_id'].nunique()),
    'top_reason_codes': test_g['reason_codes'].str.split('|').explode().value_counts().head(5).to_dict()
}
with open('work/outputs/action_playbook_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print("Saved action_queue.csv and action_playbook_metrics.json")
print(metrics_summary)

Saved action_queue.csv and action_playbook_metrics.json
{'total_pages_ranked': 7115, 'unique_clients_in_queue': 8, 'top_reason_codes': {'general_refresh_review': 3826, 'declining_with_demand': 2422, 'low_ctr_visible_page': 1922, 'stale_visible_page': 3}}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.